# Grade a plot without looking at it

A grader cannot look at the picture a student's code draws. It reads the numbers and
settings behind it instead: the data of each line, its marker, line style and color, the
labels, the bars. Here you write two such checks and run them on figures drawn right and
drawn in several wrong ways, the way an autograder would.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import same_color

# The task being graded: the number of handshakes among n people, n = 2, ..., 9.
n = np.arange(2, 10)
handshakes = n * (n - 1) // 2
print("n:         ", n)
print("handshakes:", handshakes)

## Part 1 (coding task): Check a styled line

The task asked for one line through the points `(n, handshakes)`, with a circle marker
`"o"` at each point, a dotted line `":"`, the color blue, and an x label, a y label and a
title (any text, but not empty).

Write `line_problems(ax, x, y)`. It returns a list of the requirements that `ax` fails,
using these names, in this order:

- `"lines"`: the Axes does not hold exactly one line. In that case return `["lines"]`
  alone, since there is no single line to check.
- `"data"`: the line's x data is not `x` or its y data is not `y`.
- `"marker"`, `"linestyle"`, `"color"`: the line's marker, line style or color is wrong.
- `"labels"`: the x label, the y label or the title is empty.

A figure drawn exactly as asked returns `[]`.

In [ ]:
def line_problems(ax, x, y):
    """The names of the requirements the Axes fails, in the order listed above."""
    # TODO
    raise NotImplementedError

In [ ]:
def drawn(draw, title="Handshakes among n people", xlabel="people", ylabel="handshakes"):
    fig, ax = plt.subplots()
    draw(ax)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    plt.close(fig)
    return ax

x, y = n, handshakes
cases = [
    ("the long form", drawn(lambda ax: ax.plot(x, y, marker="o", linestyle=":", color="blue")), []),
    ("the short form 'bo:'", drawn(lambda ax: ax.plot(x, y, "bo:")), []),
    ("the color as '#0000ff'", drawn(lambda ax: ax.plot(x, y, marker="o", linestyle=":", color="#0000ff")), []),
    ("y given without x", drawn(lambda ax: ax.plot(y, "bo:")), ["data"]),
    ("every y one too high", drawn(lambda ax: ax.plot(x, y + 1, "bo:")), ["data"]),
    ("only the first five points", drawn(lambda ax: ax.plot(x[:5], y[:5], "bo:")), ["data"]),
    ("a solid line", drawn(lambda ax: ax.plot(x, y, "bo-")), ["linestyle"]),
    ("red squares", drawn(lambda ax: ax.plot(x, y, "rs:")), ["marker", "color"]),
    ("no title", drawn(lambda ax: ax.plot(x, y, "bo:"), title=""), ["labels"]),
    ("no x label", drawn(lambda ax: ax.plot(x, y, "bo:"), xlabel=""), ["labels"]),
    ("no y label", drawn(lambda ax: ax.plot(x, y, "bo:"), ylabel=""), ["labels"]),
    ("the line drawn twice", drawn(lambda ax: [ax.plot(x, y, "bo:"), ax.plot(x, y, "bo:")]), ["lines"]),
]
for name, ax, expected in cases:
    got = line_problems(ax, x, y)
    assert got == expected, f"for {name} the answer should be {expected}, got {got}"
print("Part 1 passed")

## Part 2 (coding task): Check a histogram

A second task gives 200 exam scores and bin edges 10, 20, ..., 90 and asks for a histogram
of the counts on exactly those bins.

Write `bars_match(ax, values, edges)`. It returns `True` when the bars in `ax.patches`
are exactly that histogram: one bar per bin, in order, each starting at its bin's left
edge, as wide as the bin, and as tall as the bin's count from
`np.histogram(values, bins=edges)`. Otherwise it returns `False`, and it never raises.

A bar is a rectangle: `p.get_x()` is its left edge, `p.get_width()` its width and
`p.get_height()` its height. Compare with `np.allclose`, since the positions are floats.

In [ ]:
def bars_match(ax, values, edges):
    """True when ax.patches are the histogram of values on exactly these edges."""
    # TODO
    raise NotImplementedError

In [ ]:
gen = np.random.default_rng(3)
scores = np.round(gen.normal(52, 13, 200), 1)
edges = np.arange(10, 100, 10)
counts, _ = np.histogram(scores, bins=edges)

def bars(draw):
    fig, ax = plt.subplots()
    draw(ax)
    plt.close(fig)
    return ax

cases = [
    ("ax.hist on the given edges", bars(lambda ax: ax.hist(scores, bins=edges)), True),
    ("ax.bar at the bin centers, one bin wide", bars(lambda ax: ax.bar(edges[:-1] + 5, counts, width=10)), True),
    ("8 equal bins from the lowest score to the highest", bars(lambda ax: ax.hist(scores, bins=8)), False),
    ("densities instead of counts", bars(lambda ax: ax.hist(scores, bins=edges, density=True)), False),
    ("the right heights centered on the left edges", bars(lambda ax: ax.bar(edges[:-1], counts, width=10)), False),
    ("the right heights, bars 8 wide", bars(lambda ax: ax.bar(edges[:-1], counts, width=8, align="edge")), False),
    ("the last bin left out", bars(lambda ax: ax.hist(scores, bins=edges[:-1])), False),
    ("the histogram drawn twice", bars(lambda ax: [ax.hist(scores, bins=edges), ax.hist(scores, bins=edges)]), False),
]
for name, ax, expected in cases:
    got = bars_match(ax, scores, edges)
    assert isinstance(got, (bool, np.bool_)), f"return True or False, got {got!r}"
    assert got == expected, f"for {name} the answer should be {expected}, got {got}"
print("Part 2 passed")

## Solution

### Part 1: Check a styled line

In [ ]:
def line_problems(ax, x, y):
    """The names of the requirements the Axes fails, in the order listed above."""
    if len(ax.lines) != 1:
        return ["lines"]
    line = ax.lines[0]
    problems = []
    xs, ys = np.asarray(line.get_xdata(), dtype=float), np.asarray(line.get_ydata(), dtype=float)
    same_shape = xs.shape == np.shape(x) and ys.shape == np.shape(y)
    if not (same_shape and np.allclose(xs, x) and np.allclose(ys, y)):
        problems.append("data")
    if line.get_marker() != "o":
        problems.append("marker")
    if line.get_linestyle() != ":":
        problems.append("linestyle")
    if not same_color(line.get_color(), "blue"):
        problems.append("color")
    if not (ax.get_xlabel() and ax.get_ylabel() and ax.get_title()):
        problems.append("labels")
    return problems

**Why it works.** `ax.lines` holds every line in the order it was drawn, so checking its length
first means the other checks always read the one line the task asked for. The style is
compared through what matplotlib stored, not through what the student typed: the short
form `"bo:"` is split into color `"b"`, marker `"o"` and line style `":"`, and
`get_color()` gives back `"b"` or `"#0000ff"` exactly as written. Comparing it with
`== "blue"` would fail two correct figures, while `same_color` compares the colors
themselves. The data check reads x as well as y: `ax.plot(y, "bo:")` draws the right
heights at x = 0, ..., 7, so a check of y alone would pass a plot that puts every point
two places to the left. Comparing shapes before `np.allclose` turns a line with too few
points into a failed check rather than an error.

### Part 2: Check a histogram

In [ ]:
def bars_match(ax, values, edges):
    """True when ax.patches are the histogram of values on exactly these edges."""
    edges = np.asarray(edges, dtype=float)
    counts, _ = np.histogram(values, bins=edges)
    bars = ax.patches
    if len(bars) != len(counts):
        return False
    lefts = np.array([p.get_x() for p in bars])
    widths = np.array([p.get_width() for p in bars])
    heights = np.array([p.get_height() for p in bars])
    return bool(
        np.allclose(lefts, edges[:-1])
        and np.allclose(widths, np.diff(edges))
        and np.allclose(heights, counts)
    )

**Why it works.** `np.histogram` gives the counts `ax.hist` would draw on the same edges, so
the heights have a reference to compare with. Heights alone are not enough: two of the
wrong figures have exactly the right heights, one shifted half a bin to the left (`ax.bar`
centers each bar on its x value unless told `align="edge"`) and one with gaps between
narrower bars. Checking the left edges and widths as well pins every bar to its bin, and
then it does not matter whether the student called `ax.hist` or `ax.bar`, which is how it
should be. Comparing the number of bars first keeps `np.allclose` from failing on arrays
of different lengths, and catches a histogram drawn twice.